# 02 Bronze FHIR Ingestion

## Purpose

In this notebook, we ingest raw Synthea FHIR R4 JSON bundle files from Databricks Volume storage and create Bronze Delta tables.

## What We Are Doing

We will:
1. Read raw FHIR JSON files from the raw volume.
2. Explode FHIR bundle entries.
3. Extract individual FHIR resources.
4. Separate major healthcare resources.
5. Save them as Bronze Delta tables.

## Why We Are Doing This

FHIR files are nested JSON bundles. They are not flat tables.

The Bronze layer keeps raw resources in a queryable Delta format while preserving the original FHIR structure.

## Expected Final Output

Bronze tables will be created inside:

`healthcare_catalog.bronze`

Expected tables:
- patient_raw
- encounter_raw
- observation_raw
- condition_raw
- medication_raw

In [0]:
from pyspark.sql.functions import *

## Step 1 — Load Raw FHIR JSON Files

### What We Are Doing

We are reading raw Synthea FHIR R4 JSON files from the Databricks Volume.

### Why We Are Doing This

The uploaded files are raw FHIR bundles. Before creating tables, Spark must load them into a DataFrame.

We use `multiLine=True` because FHIR JSON files usually span multiple lines.

### Expected Output

A Spark DataFrame named `raw_fhir_df` will be created.

This DataFrame represents raw FHIR bundle files.

In [0]:
raw_fhir_df = spark.read.option("multiLine", True).json(
    "/Volumes/healthcare_catalog/bronze/raw_fhir_files/*.json"
)

print("Raw FHIR JSON files loaded successfully.")

Raw FHIR JSON files loaded successfully.


## Step 2 — Inspect Raw FHIR Schema

### What We Are Doing

We are printing the schema of the raw FHIR DataFrame.

### Why We Are Doing This

FHIR data is nested and complex. Before transforming it, we need to understand the structure.

### Expected Output

You should see:
- `entry`
- `resourceType`
- `type`

The most important field is `entry`, because it contains all healthcare resources inside each FHIR bundle.

In [0]:
raw_fhir_df.printSchema()

root
 |-- entry: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- fullUrl: string (nullable = true)
 |    |    |-- request: struct (nullable = true)
 |    |    |    |-- ifNoneExist: string (nullable = true)
 |    |    |    |-- method: string (nullable = true)
 |    |    |    |-- url: string (nullable = true)
 |    |    |-- resource: struct (nullable = true)
 |    |    |    |-- abatementDateTime: string (nullable = true)
 |    |    |    |-- active: boolean (nullable = true)
 |    |    |    |-- activity: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |

## Step 3 — Explode FHIR Bundle Entries

### What We Are Doing

We are exploding the `entry` array from each FHIR bundle.

### Why We Are Doing This

Each FHIR bundle contains many resources inside `entry[]`.

For example:
- Patient
- Encounter
- Observation
- Condition
- MedicationRequest

Spark needs one row per resource before we can create Bronze tables.

### Expected Output

A DataFrame named `fhir_entries_df`.

Each row now represents one entry from a FHIR bundle.

In [0]:
fhir_entries_df = raw_fhir_df.select(
    explode(col("entry")).alias("entry")
)

print("FHIR bundle entries exploded successfully.")

FHIR bundle entries exploded successfully.


## Step 4 — Extract Individual FHIR Resources

### What We Are Doing

We are extracting:
- `fullUrl`
- `resourceType`
- `resource`

from each exploded bundle entry.

### Why We Are Doing This

`resourceType` tells us what kind of healthcare resource each row contains.

Examples:
- Patient
- Encounter
- Observation
- Condition
- MedicationRequest

This allows us to split the raw FHIR resources into separate Bronze tables.

### Expected Output

A DataFrame named `fhir_resources_df`.

Each row represents one FHIR healthcare resource.

In [0]:
fhir_resources_df = fhir_entries_df.select(
    col("entry.fullUrl").alias("fullUrl"),
    col("entry.resource.resourceType").alias("resourceType"),
    col("entry.resource").alias("resource")
)

print("FHIR resources extracted successfully.")

FHIR resources extracted successfully.


## Step 5 — Count Resource Types

### What We Are Doing

We are counting how many records exist for each FHIR resource type.

### Why We Are Doing This

Before saving Bronze tables, we need to validate which healthcare resources exist in the dataset.

### Expected Output

A table showing resource counts such as:
- Patient
- Encounter
- Observation
- Condition
- MedicationRequest
- Procedure
- Claim
- DiagnosticReport

This confirms that our FHIR extraction worked correctly.

In [0]:
display(
    fhir_resources_df.groupBy("resourceType").count()
)

resourceType,count
MedicationRequest,24256
Immunization,8100
ExplanationOfBenefit,27812
CareTeam,1831
Patient,555
Observation,131703
Provenance,555
DocumentReference,27812
Procedure,38528
Device,44


## Step 6 — Create Bronze Patient DataFrame

### What We Are Doing

We are filtering only Patient resources from the extracted FHIR resources.

### Why We Are Doing This

Patient resources contain demographic and identity information.

This is the foundation of the healthcare platform.

### Expected Output

A DataFrame named `patient_bronze_df`.

It should contain only rows where `resourceType = Patient`.

In [0]:
patient_bronze_df = fhir_resources_df.filter(
    col("resourceType") == "Patient"
)

print("Patient Bronze DataFrame created successfully.")

Patient Bronze DataFrame created successfully.


## Step 7 — Save Bronze Patient Table

### What We Are Doing

We are saving the Patient Bronze DataFrame as a Delta table.

### Why We Are Doing This

Delta tables are queryable, reliable, and reusable.

This table becomes the raw Patient table in the Bronze layer.

### Expected Output

A Delta table will be created:

`healthcare_catalog.bronze.patient_raw`

In [0]:
patient_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.patient_raw")

print("Bronze patient_raw table saved successfully.")

Bronze patient_raw table saved successfully.


## Step 8 — Create Bronze Encounter DataFrame

### What We Are Doing
We are filtering only `Encounter` resources from the extracted FHIR resources.

### Why We Are Doing This
Encounter resources represent patient visits, hospital encounters, outpatient visits, and care events.

This table will later help us calculate:
- patient visit history
- length of stay
- readmission patterns
- encounter frequency

### Expected Output
A DataFrame named `encounter_bronze_df`.

It should contain only rows where `resourceType = Encounter`.

In [0]:
encounter_bronze_df = fhir_resources_df.filter(
    col("resourceType") == "Encounter"
)

print("Encounter Bronze DataFrame created successfully.")

Encounter Bronze DataFrame created successfully.


## Step 9 — Save Bronze Encounter Table

### What We Are Doing
We are saving the raw Encounter resources as a Delta table.

### Why We Are Doing This
The Bronze layer stores raw healthcare resources in Delta format for repeatable downstream processing.

### Expected Output
A Delta table will be created:

`healthcare_catalog.bronze.encounter_raw`

In [0]:
encounter_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.encounter_raw")

print("Bronze encounter_raw table saved successfully.")

Bronze encounter_raw table saved successfully.


## Step 10 — Create Bronze Observation DataFrame

### What We Are Doing
We are filtering only `Observation` resources from the FHIR resources.

### Why We Are Doing This
Observation resources contain clinical measurements such as:
- lab results
- vital signs
- BMI
- blood pressure
- glucose
- cholesterol

This will become one of the most important inputs for analytics and ML.

### Expected Output
A DataFrame named `observation_bronze_df`.

It should contain only rows where `resourceType = Observation`.

In [0]:
observation_bronze_df = fhir_resources_df.filter(
    col("resourceType") == "Observation"
)

print("Observation Bronze DataFrame created successfully.")

Observation Bronze DataFrame created successfully.


## Step 11 — Save Bronze Observation Table

### What We Are Doing
We are saving raw Observation resources as a Delta table.

### Why We Are Doing This
Observations are large and clinically rich. Saving them separately makes downstream cleaning and feature engineering easier.

### Expected Output
A Delta table will be created:

`healthcare_catalog.bronze.observation_raw`

In [0]:
observation_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.observation_raw")

print("Bronze observation_raw table saved successfully.")

Bronze observation_raw table saved successfully.


In [0]:
# Final Verification

spark.sql("""
SHOW TABLES IN healthcare_catalog.bronze
""").show(truncate=False)

+--------+---------------+-----------+
|database|tableName      |isTemporary|
+--------+---------------+-----------+
|bronze  |encounter_raw  |false      |
|bronze  |observation_raw|false      |
|bronze  |patient_raw    |false      |
+--------+---------------+-----------+



## Step 12 — Create Bronze Condition DataFrame

### What We Are Doing
We are filtering only `Condition` resources from the extracted FHIR resources.

### Why We Are Doing This
Condition resources represent patient diagnoses and clinical problems.

This table will later support:
- chronic disease analytics
- diabetes risk modeling
- comorbidity features
- population health analysis

### Expected Output
A DataFrame named `condition_bronze_df`.

It should contain only rows where `resourceType = Condition`.

In [0]:
condition_bronze_df = fhir_resources_df.filter(
    col("resourceType") == "Condition"
)

print("Condition Bronze DataFrame created successfully.")

Condition Bronze DataFrame created successfully.


## Step 13 — Save Bronze Condition Table

### What We Are Doing
We are saving raw Condition resources as a Delta table.

### Why We Are Doing This
Diagnoses are central to healthcare analytics and clinical ML.

The Bronze layer preserves the original FHIR structure before Silver cleaning.

### Expected Output
A Delta table will be created:

`healthcare_catalog.bronze.condition_raw`

In [0]:
condition_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.condition_raw")

print("Bronze condition_raw table saved successfully.")

Bronze condition_raw table saved successfully.


## Step 14 — Create Bronze MedicationRequest DataFrame

### What We Are Doing
We are filtering only `MedicationRequest` resources from the extracted FHIR resources.

### Why We Are Doing This
MedicationRequest resources represent prescribed medications.

This table will later support:
- medication usage analytics
- adherence-related features
- polypharmacy analysis
- chronic disease treatment patterns

### Expected Output
A DataFrame named `medication_bronze_df`.

It should contain only rows where `resourceType = MedicationRequest`.

In [0]:
medication_bronze_df = fhir_resources_df.filter(
    col("resourceType") == "MedicationRequest"
)

print("MedicationRequest Bronze DataFrame created successfully.")

MedicationRequest Bronze DataFrame created successfully.


## Step 15 — Save Bronze MedicationRequest Table

### What We Are Doing
We are saving raw MedicationRequest resources as a Delta table.

### Why We Are Doing This
Medication data is important for clinical intelligence, risk modeling, and patient care analytics.

### Expected Output
A Delta table will be created:

`healthcare_catalog.bronze.medication_request_raw`

In [0]:
medication_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.medication_request_raw")

print("Bronze medication_request_raw table saved successfully.")

Bronze medication_request_raw table saved successfully.


## Step 16 — Verify Bronze Tables

### What We Are Doing
We are listing all tables inside the Bronze schema.

### Why We Are Doing This
This confirms that our raw FHIR resources were successfully separated and saved as Delta tables.

### Expected Output
You should see:
- patient_raw
- encounter_raw
- observation_raw
- condition_raw
- medication_request_raw

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.bronze
""").show(truncate=False)

+--------+----------------------+-----------+
|database|tableName             |isTemporary|
+--------+----------------------+-----------+
|bronze  |condition_raw         |false      |
|bronze  |encounter_raw         |false      |
|bronze  |medication_request_raw|false      |
|bronze  |observation_raw       |false      |
|bronze  |patient_raw           |false      |
+--------+----------------------+-----------+



# Bronze Procedure Ingestion

## What We Are Doing
We are creating the missing Bronze Procedure table.

## Why We Are Doing This
The Silver Procedure notebook cannot run until the raw Procedure resource exists in the Bronze layer.

## Expected Output
A new Bronze table:

`healthcare_catalog.bronze.procedure_raw`

In [0]:
display(dbutils.fs.ls("/Volumes/healthcare_catalog/bronze/raw_fhir_files"))

path,name,size,modificationTime
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json,Abdul218_Harris789_b0a06ead-cc42-aa48-dad6-841d4aa679fa.json,1951150,1778597864000
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Abe604_Schmidt332_ccfc4db2-2026-7adb-3db0-33f3828140bb.json,Abe604_Schmidt332_ccfc4db2-2026-7adb-3db0-33f3828140bb.json,1615835,1778597863000
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Adelaida985_DuBuque211_31a2e8ec-69fc-8a71-3ab6-36cbdd508713.json,Adelaida985_DuBuque211_31a2e8ec-69fc-8a71-3ab6-36cbdd508713.json,2111611,1778597864000
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Adriana394_Hintz995_71a8b156-760b-df6b-859e-eefc7932a526.json,Adriana394_Hintz995_71a8b156-760b-df6b-859e-eefc7932a526.json,810415,1778597861000
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Agueda283_Gerhold939_76b289fd-e825-734c-8446-316f59643593.json,Agueda283_Gerhold939_76b289fd-e825-734c-8446-316f59643593.json,1910506,1778597864000
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Ahmed109_O'Reilly797_92fb7efc-5cfd-f8d3-927b-42f8ee099531.json,Ahmed109_O'Reilly797_92fb7efc-5cfd-f8d3-927b-42f8ee099531.json,1076052,1778597862000
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Aimee901_Hudson301_81aa7647-779f-fd6b-94cf-782e606efeb2.json,Aimee901_Hudson301_81aa7647-779f-fd6b-94cf-782e606efeb2.json,738028,1778597861000
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Akiko835_Pfannerstill264_346a1435-2455-914f-c287-7b88052d05db.json,Akiko835_Pfannerstill264_346a1435-2455-914f-c287-7b88052d05db.json,3058596,1778597866000
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Alaine226_Willms744_1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4.json,Alaine226_Willms744_1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4.json,1870481,1778597864000
dbfs:/Volumes/healthcare_catalog/bronze/raw_fhir_files/Alberta625_Waters156_97899f1d-9c3b-2b90-17b8-400c11ab8f0f.json,Alberta625_Waters156_97899f1d-9c3b-2b90-17b8-400c11ab8f0f.json,1987859,1778597864000


## Step 2 — Read All Raw FHIR Bundle JSON Files

### What We Are Doing
We are reading all uploaded raw FHIR JSON files from the Databricks Volume.

### Why We Are Doing This
Your Synthea files are patient-level FHIR Bundle JSON files.  
Each file contains many resource types such as Patient, Encounter, Observation, Condition, Procedure, Claim, and others.

### Expected Output
A raw FHIR Bundle DataFrame named `raw_fhir_df`.

In [0]:
raw_fhir_df = spark.read.option("multiLine", True).json(
    "/Volumes/healthcare_catalog/bronze/raw_fhir_files/*.json"
)

print("Raw FHIR bundle files loaded successfully.")

Raw FHIR bundle files loaded successfully.


## Step 3 — Explode Bundle Entries

### What We Are Doing
We are exploding the FHIR Bundle `entry` array.

### Why We Are Doing This
Each FHIR Bundle file contains many resources inside:

`entry[]`

We need to convert each resource into its own Spark row.

### Expected Output
A DataFrame where each row represents one FHIR resource.

In [0]:
fhir_entries_df = raw_fhir_df.select(
    explode(col("entry")).alias("entry")
)

fhir_resources_df = fhir_entries_df.select(
    col("entry.fullUrl").alias("fullUrl"),
    col("entry.resource.resourceType").alias("resourceType"),
    col("entry.resource").alias("resource")
)

print("FHIR Bundle entries exploded successfully.")

FHIR Bundle entries exploded successfully.


## Step 4 — Filter Procedure Resources

### What We Are Doing
We are filtering only FHIR Procedure resources.

### Why We Are Doing This
The Bronze Procedure table should contain only raw Procedure resources.

### Expected Output
A DataFrame named `procedure_raw_df`.

In [0]:
procedure_raw_df = fhir_resources_df.filter(
    col("resourceType") == "Procedure"
)

print("Procedure Bronze DataFrame created successfully.")

display(
    procedure_raw_df.select("resourceType").groupBy("resourceType").count()
)

Procedure Bronze DataFrame created successfully.


resourceType,count
Procedure,38528


## Step 5 — Save Bronze Procedure Table

### What We Are Doing
We are saving raw Procedure resources into the Bronze layer.

### Why We Are Doing This
The Silver Procedure notebook needs this Bronze input table.

### Expected Output
A new Delta table:

`healthcare_catalog.bronze.procedure_raw`

In [0]:
procedure_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.procedure_raw")

print("Bronze procedure_raw table saved successfully.")

Bronze procedure_raw table saved successfully.


In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.bronze
""").show(truncate=False)

+--------+----------------------+-----------+
|database|tableName             |isTemporary|
+--------+----------------------+-----------+
|bronze  |condition_raw         |false      |
|bronze  |encounter_raw         |false      |
|bronze  |medication_request_raw|false      |
|bronze  |observation_raw       |false      |
|bronze  |patient_raw           |false      |
|bronze  |procedure_raw         |false      |
+--------+----------------------+-----------+



# Bronze Immunization Ingestion

## Purpose
Create raw Bronze table for FHIR Immunization resources.

## Expected Output
`healthcare_catalog.bronze.immunization_raw`

In [0]:
from pyspark.sql.functions import *

In [0]:
raw_fhir_df = spark.read.option("multiLine", True).json(
    "/Volumes/healthcare_catalog/bronze/raw_fhir_files/*.json"
)

fhir_entries_df = raw_fhir_df.select(
    explode(col("entry")).alias("entry")
)

fhir_resources_df = fhir_entries_df.select(
    col("entry.fullUrl").alias("fullUrl"),
    col("entry.resource.resourceType").alias("resourceType"),
    col("entry.resource").alias("resource")
)

immunization_raw_df = fhir_resources_df.filter(
    col("resourceType") == "Immunization"
)

display(immunization_raw_df.groupBy("resourceType").count())

resourceType,count
Immunization,8100


In [0]:
immunization_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.immunization_raw")

print("Bronze immunization_raw table created successfully.")

Bronze immunization_raw table created successfully.


# Bronze CarePlan Ingestion

## Purpose
Create raw Bronze table for FHIR CarePlan resources.

## Expected Output
`healthcare_catalog.bronze.careplan_raw`

In [0]:
raw_fhir_df = spark.read.option("multiLine", True).json(
    "/Volumes/healthcare_catalog/bronze/raw_fhir_files/*.json"
)

fhir_entries_df = raw_fhir_df.select(
    explode(col("entry")).alias("entry")
)

fhir_resources_df = fhir_entries_df.select(
    col("entry.fullUrl").alias("fullUrl"),
    col("entry.resource.resourceType").alias("resourceType"),
    col("entry.resource").alias("resource")
)

careplan_raw_df = fhir_resources_df.filter(
    col("resourceType") == "CarePlan"
)

display(careplan_raw_df.groupBy("resourceType").count())

resourceType,count
CarePlan,1831


In [0]:
careplan_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.careplan_raw")

print("Bronze careplan_raw table created successfully.")

Bronze careplan_raw table created successfully.


# Bronze AllergyIntolerance Ingestion

## Purpose
Create raw Bronze table for FHIR AllergyIntolerance resources.

## Expected Output
`healthcare_catalog.bronze.allergy_intolerance_raw`

In [0]:
raw_fhir_df = spark.read.option("multiLine", True).json(
    "/Volumes/healthcare_catalog/bronze/raw_fhir_files/*.json"
)

fhir_entries_df = raw_fhir_df.select(
    explode(col("entry")).alias("entry")
)

fhir_resources_df = fhir_entries_df.select(
    col("entry.fullUrl").alias("fullUrl"),
    col("entry.resource.resourceType").alias("resourceType"),
    col("entry.resource").alias("resource")
)

allergy_intolerance_raw_df = fhir_resources_df.filter(
    col("resourceType") == "AllergyIntolerance"
)

display(allergy_intolerance_raw_df.groupBy("resourceType").count())

resourceType,count
AllergyIntolerance,499


In [0]:
allergy_intolerance_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.allergy_intolerance_raw")

print("Bronze allergy_intolerance_raw table created successfully.")

Bronze allergy_intolerance_raw table created successfully.


# Bronze Claim Ingestion

## Purpose
Create raw Bronze table for FHIR Claim resources.

## Expected Output
`healthcare_catalog.bronze.claim_raw`

In [0]:
raw_fhir_df = spark.read.option("multiLine", True).json(
    "/Volumes/healthcare_catalog/bronze/raw_fhir_files/*.json"
)

fhir_entries_df = raw_fhir_df.select(
    explode(col("entry")).alias("entry")
)

fhir_resources_df = fhir_entries_df.select(
    col("entry.fullUrl").alias("fullUrl"),
    col("entry.resource.resourceType").alias("resourceType"),
    col("entry.resource").alias("resource")
)

claim_raw_df = fhir_resources_df.filter(
    col("resourceType") == "Claim"
)

display(claim_raw_df.groupBy("resourceType").count())

resourceType,count
Claim,52068


In [0]:
claim_raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.bronze.claim_raw")

print("Bronze claim_raw table created successfully.")

Bronze claim_raw table created successfully.


In [0]:
#Final Bronze Verification Notebook/Cell
spark.sql("""
SHOW TABLES IN healthcare_catalog.bronze
""").show(truncate=False)

+--------+-----------------------+-----------+
|database|tableName              |isTemporary|
+--------+-----------------------+-----------+
|bronze  |allergy_intolerance_raw|false      |
|bronze  |careplan_raw           |false      |
|bronze  |claim_raw              |false      |
|bronze  |condition_raw          |false      |
|bronze  |encounter_raw          |false      |
|bronze  |immunization_raw       |false      |
|bronze  |medication_request_raw |false      |
|bronze  |observation_raw        |false      |
|bronze  |patient_raw            |false      |
|bronze  |procedure_raw          |false      |
+--------+-----------------------+-----------+

